# 🌋 Recreating the Seismic "Matryoshka" (Mantle Tomography nesting shells)

This notebook demonstrates how to build the nested Seismic "Matryoshka" model (Section 3.2.3 of Koelemeijer & Winterbourne 2021).

### 🌎 Scientific Context
Seismic tomography models are three-dimensional maps of how fast seismic waves propagate through the Earth's mantle. Variations in velocity correlate with temperature anomalies, telling us where hot rock is rising and cold rock is sinking:
- **Lithosphere (50 km)**: Fast velocities represent cold, rigid continental cratons; slow velocities represent hot ocean ridges. Dominant mineral: **olivine** (represented in **green**).
- **Transition Zone (660 km)**: Marked by mineral phase changes where slabs subduct. Dominant mineral: **ringwoodite** (represented in **blue**).
- **Lower Mantle (2,850 km)**: Dominated by two massive structures under Africa and the Pacific called Large Low-Velocity Provinces (LLVPs). Dominant mineral: **bridgmanite** (represented in **gold/brown**).

To represent this, we map velocity anomalies to physical topography (low velocity = hot = elevated topography; high velocity = cold = depressed topography) and nesting-fit them together in concentric shells.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    GridColourer,
    ConstantColourer
)

## Step 2: Load Datasets

We use the preloaded Core-Mantle Boundary slice `s40_depth_slice_2850.grd` to run the code. 

*(To download the real 50 km and 660 km grids, check the [Data Sourcing Guide](../../user_guide/8_where_to_get_data.md) and replace the paths below).* 

For this demo, we reuse the 2850 km grid with added offsets/perturbations to act as placeholders for the shallower mantle depths.

In [ ]:
# Load the seismic tomography grid
tomo_grid = GeographicGrid.from_netcdf(
    "../../inputs/s40_depth_slice_2850.grd", 'y', 'x', 'z'
)

# Create simulated offset grids for 50 km and 660 km depths (placeholders)
grid_50 = GeographicGrid(lats=tomo_grid.lats, lons=tomo_grid.lons, grid=tomo_grid.grid * -0.8)
grid_660 = GeographicGrid(lats=tomo_grid.lats, lons=tomo_grid.lons, grid=np.roll(tomo_grid.grid, 90, axis=1))
grid_2850 = tomo_grid

print("Grids loaded/simulated successfully.")

## Step 3: Recreate the Three Shell Layers

We configure three concentric models:
1. **Outer Shell (50 km)**: Radius 40 mm, hollow cavity 28 mm. Colormap: `YlGn` (Green).
2. **Middle Shell (660 km)**: Radius 27.5 mm, hollow cavity 18 mm. Colormap: `Blues` (Blue).
3. **Inner Shell (2,850 km)**: Radius 17.5 mm, solid. Colormap: `YlOrBr` (Gold/Brown).

### 📐 Scaling Velocity Anomalies to Millimeters
Seismic velocity anomalies in our datasets range roughly from $-2\%$ to $+2\%$. Because these are unitless percentage values rather than physical dimensions, we set the scale factor directly in Python. A scale factor of `0.75` translates each $1\%$ anomaly to $0.75\text{ mm}$ of physical surface displacement (meaning a $2\%$ anomaly results in a $1.5\text{ mm}$ peak or valley on the print).

In [ ]:
os.makedirs('../../outputs', exist_ok=True)
scale_val = 0.75  # 0.75 mm physical displacement per percent anomaly

# --- 1. OUTER SHELL (50 km - Olivine Green) ---
shell_50 = GlobeModel(n_points=5000, radius=40.0, hollow=True, inner_ratio=0.7) # 28mm inner
shell_50.outer.displace(GridDisplacer(grid_50), scale=scale_val)
# Color: green outer, gray inner
shell_50.outer.colour(GridColourer(grid_50, colormap='YlGn', vmin=-2.0, vmax=2.0), selection='outward_facing')
shell_50.outer.colour(ConstantColourer([0.6, 0.6, 0.6]), selection='inward_facing')
# Magnets and export
shell_50.configure_magnets(diameter=5.0, height=2.0, n_magnets=3, position=0.0, add_bosses=True)
shell_50.export_hemispheres("../../outputs/matryoshka_50_top.obj", "../../outputs/matryoshka_50_bottom.obj")

# --- 2. MIDDLE SHELL (660 km - Ringwoodite Blue) ---
shell_660 = GlobeModel(n_points=4000, radius=27.5, hollow=True, inner_ratio=0.65) # 18mm inner
shell_660.outer.displace(GridDisplacer(grid_660), scale=scale_val)
# Color: blue outer, gray inner
shell_660.outer.colour(GridColourer(grid_660, colormap='Blues', vmin=-2.0, vmax=2.0), selection='outward_facing')
shell_660.outer.colour(ConstantColourer([0.6, 0.6, 0.6]), selection='inward_facing')
# Magnets and export
shell_660.configure_magnets(diameter=4.0, height=1.5, n_magnets=3, position=0.0, add_bosses=True)
shell_660.export_hemispheres("../../outputs/matryoshka_660_top.obj", "../../outputs/matryoshka_660_bottom.obj")

# --- 3. INNER SHELL (2,850 km - Bridgmanite Gold) ---
shell_2850 = GlobeModel(n_points=3000, radius=17.5, hollow=False)
shell_2850.outer.displace(GridDisplacer(grid_2850), scale=scale_val)
# Color: gold outer
shell_2850.outer.colour(GridColourer(grid_2850, colormap='YlOrBr', vmin=-2.0, vmax=2.0))
# Note: add_bosses=True is used even for solid spheres. The bosses merge with the solid core
shell_2850.configure_magnets(diameter=3.0, height=1.5, n_magnets=3, position=0.0, add_bosses=True)
shell_2850.export_hemispheres("../../outputs/matryoshka_2850_top.obj", "../../outputs/matryoshka_2850_bottom.obj")

print("All Matryoshka shell files exported to outputs/")

## Step 4: Preview the Model in 3D

Preview the interactive 3D model (Outer Shell):

In [ ]:
shell_50.preview()